In [1]:
%pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
from sqlalchemy import create_engine, Column, Integer, String, ForeignKey, Float
from sqlalchemy.orm import declarative_base, relationship


def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "setup").exists() and (candidate / "apiserver").exists():
            return candidate
    return cwd


PROJECT_ROOT = find_project_root()
DB_PATH = PROJECT_ROOT / "setup" / "travel_planner.db"
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

# Delete old database file for a clean slate.
if DB_PATH.exists():
    DB_PATH.unlink()
    print(f"{DB_PATH.name} file has been deleted!")
else:
    print("No canonical database file to delete!")

Base = declarative_base()

# ---------------------------------------------------------
# TABLE 1: Destinations
# ---------------------------------------------------------
class Destination(Base):
    __tablename__ = 'Destinations'

    DestinationID = Column(Integer, primary_key=True)
    CityName = Column(String, nullable=False)
    Country = Column(String, nullable=False)
    Description = Column(String, nullable=False)

    activities = relationship("Destination_Activity", back_populates="destination")
    vibes = relationship("Destination_Vibe", back_populates="destination")
    cost_profile = relationship("CostProfile", back_populates="destination", uselist=False)
    weather_records = relationship("WeatherMonthly", back_populates="destination")

    def __repr__(self):
        return f"<Destination(City='{self.CityName}', Country='{self.Country}')>"

# ---------------------------------------------------------
# TABLE 2: Cost_Profiles
# ---------------------------------------------------------
class CostProfile(Base):
    __tablename__ = 'Cost_Profiles'

    CostID = Column(Integer, primary_key=True)
    DestinationID = Column(Integer, ForeignKey('Destinations.DestinationID'), unique=True, nullable=False)
    BudgetLevel = Column(String, nullable=False)

    destination = relationship("Destination", back_populates="cost_profile")

# ---------------------------------------------------------
# TABLE 3: Weather_Monthly
# ---------------------------------------------------------
class WeatherMonthly(Base):
    __tablename__ = 'Weather_Monthly'

    WeatherID = Column(Integer, primary_key=True)
    DestinationID = Column(Integer, ForeignKey('Destinations.DestinationID'), nullable=False)
    Month = Column(Integer, nullable=False)
    AvgTempC = Column(Float)
    RainfallMM = Column(Float)

    destination = relationship("Destination", back_populates="weather_records")

# ---------------------------------------------------------
# TABLE 4: Activities
# ---------------------------------------------------------
class Activity(Base):
    __tablename__ = 'Activities'

    ActivityID = Column(Integer, primary_key=True)
    ActivityName = Column(String, nullable=False)

    destinations = relationship("Destination_Activity", back_populates="activity")

    def __repr__(self):
        return f"<Activity(Name='{self.ActivityName}')>"

# ---------------------------------------------------------
# TABLE 5: Destinations_Activities
# ---------------------------------------------------------
class Destination_Activity(Base):
    __tablename__ = 'Destinations_Activities'

    DestinationID = Column(Integer, ForeignKey('Destinations.DestinationID'), primary_key=True)
    ActivityID = Column(Integer, ForeignKey('Activities.ActivityID'), primary_key=True)
    Spotlight_Description = Column(String)

    destination = relationship("Destination", back_populates="activities")
    activity = relationship("Activity", back_populates="destinations")

# ---------------------------------------------------------
# TABLE 6: Travel_Vibes
# ---------------------------------------------------------
class Travel_Vibe(Base):
    __tablename__ = 'Travel_Vibes'

    VibeID = Column(Integer, primary_key=True)
    VibeName = Column(String, nullable=False)

    destinations = relationship("Destination_Vibe", back_populates="vibe")

    def __repr__(self):
        return f"<Vibe(Name='{self.VibeName}')>"

# ---------------------------------------------------------
# TABLE 7: Destination_Vibes
# ---------------------------------------------------------
class Destination_Vibe(Base):
    __tablename__ = 'Destination_Vibes'

    DestinationID = Column(Integer, ForeignKey('Destinations.DestinationID'), primary_key=True)
    VibeID = Column(Integer, ForeignKey('Travel_Vibes.VibeID'), primary_key=True)

    destination = relationship("Destination", back_populates="vibes")
    vibe = relationship("Travel_Vibe", back_populates="destinations")

# ---------------------------------------------------------
# BUILD THE DATABASE
# ---------------------------------------------------------
engine = create_engine(f"sqlite:///{DB_PATH.as_posix()}", echo=True)
Base.metadata.create_all(engine)
print(f"\nSuccess: {DB_PATH} and all connected tables have been created!")


No canonical database file to delete!
2026-06-01 13:46:07,358 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-01 13:46:07,359 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("Destinations")
2026-06-01 13:46:07,359 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-01 13:46:07,360 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("Destinations")
2026-06-01 13:46:07,360 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-01 13:46:07,361 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("Cost_Profiles")
2026-06-01 13:46:07,361 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-01 13:46:07,362 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("Cost_Profiles")
2026-06-01 13:46:07,362 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-01 13:46:07,363 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("Weather_Monthly")
2026-06-01 13:46:07,363 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-01 13:46:07,364 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("Weather_